<a href="https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I'm using a **Random Forest classifier**, alongside a Logistic
Regression and Decision Tree for comparison. This matches the
starter pipeline's approach and fits Lane 2 (Refresh/Content
Opportunity Scoring) because the label (is_declining_label) is
binary, and tree-based ensembles handle mixed numeric + categorical
signals (position, freshness, word count, CTR) without heavy manual
feature engineering. Random Forest is also robust to the outliers
common in impressions/traffic data.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

This uses a **client-grouped holdout split**: entire clients are
kept out of training and only appear in the test set. This is
honest for this question because pages from the same client can
share patterns (site structure, content style, niche) that the
model could memorize if it saw some of that client's pages in
training and others in testing — a plain random split would let the
model cheat by learning client-specific quirks instead of general
signal.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

Below: features are prepared, a client-grouped train/test split is
built, three models are trained, and Precision@20/@50 is compared
against the Week-4 baseline score on the same test split.

In [6]:
!git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
%cd flyRank-internship


Cloning into 'flyRank-internship'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 139 (delta 54), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.86 MiB | 9.46 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flyRank-internship/flyRank-internship


In [7]:
# If fresh session, run this first:
# !git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
# %cd flyRank-internship

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")

# Label
y = (df["trend_direction"] == "down").astype(int)

# Features
num_features = ["content_age_days", "days_since_last_update", "impressions_90d",
                 "avg_position", "ctr", "word_count"]
X = df[num_features].replace([np.inf, -np.inf], np.nan).fillna(0)

# Client-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
}

# Rebuild the Week-4 baseline score on the SAME test rows for a fair comparison
def normalize(s):
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0)
    return (s - s.min()) / (s.max() - s.min()) if s.max() != s.min() else s * 0

visibility_score = normalize(df["impressions_90d"])
freshness_risk_score = normalize(df["days_since_last_update"])
position_opportunity_score = normalize(-df["avg_position"].fillna(df["avg_position"].max()))
depth_gap_score = normalize(-df["word_count"].fillna(df["word_count"].max()))
baseline_score_all = (0.40*visibility_score + 0.30*freshness_risk_score
                       + 0.25*position_opportunity_score + 0.05*depth_gap_score)
baseline_test_scores = baseline_score_all.iloc[test_idx].values

results = []
for k in (20, 50):
    baseline_p = precision_at_k(baseline_test_scores, y_test.values, k)
    results.append({"model": "Baseline (Week 4)", "k": k, "precision_at_k": round(baseline_p, 3)})

for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    for k in (20, 50):
        p = precision_at_k(probs, y_test.values, k)
        results.append({"model": name, "k": k, "precision_at_k": round(p, 3)})

results_df = pd.DataFrame(results)
print(results_df.pivot(index="model", columns="k", values="precision_at_k"))

k                      20    50
model                          
Baseline (Week 4)    0.45  0.50
Decision Tree        0.60  0.60
Logistic Regression  0.65  0.66
Random Forest        0.60  0.68


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

Below, the Random Forest's top-20 predictions are compared against
the actual labels to find where the model is wrong, and its feature
importances are shown to see what it leans on most.

In [8]:
rf = models["Random Forest"]
rf_probs = rf.predict_proba(X_test)[:, 1]

test_view = X_test.copy()
test_view["actual_declining"] = y_test.values
test_view["rf_score"] = rf_probs
test_view = test_view.sort_values("rf_score", ascending=False)

print("Top 20 by Random Forest score (1 = actually declining):")
print(test_view.head(20)[["rf_score", "actual_declining"]])

wrong_in_top20 = test_view.head(20)[test_view.head(20)["actual_declining"] == 0]
print(f"\nFalse positives in top 20: {len(wrong_in_top20)} out of 20")

importances = pd.Series(rf.feature_importances_, index=num_features).sort_values(ascending=False)
print("\nFeature importances (what the model leans on):")
print(importances)

Top 20 by Random Forest score (1 = actually declining):
       rf_score  actual_declining
166       0.990                 1
2357      0.990                 0
22526     0.985                 0
4480      0.980                 0
310       0.980                 1
9504      0.980                 1
12069     0.980                 0
10731     0.980                 1
24890     0.975                 1
5399      0.975                 0
2150      0.975                 0
11664     0.970                 1
15732     0.970                 1
17611     0.970                 1
18989     0.970                 1
6957      0.970                 1
11877     0.965                 0
370       0.965                 1
11433     0.965                 0
12246     0.965                 1

False positives in top 20: 8 out of 20

Feature importances (what the model leans on):
impressions_90d           0.275449
avg_position              0.249109
content_age_days          0.161429
word_count                0.156026
ct

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.